# 11 — Random Forest Baselines

Full ablation set: RNA-only, protein-only, RNA+drug, protein+drug. Fixed hyperparameters, top-variance feature selection computed from train-fold cell lines only (no leakage), evaluated on all 4 test splits per fold with `evaluateMT`.

## Environment setup (Colab or local)

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q rdkit
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")


## Imports and config

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import pearsonr, spearmanr, linregress
from sklearn.metrics import r2_score, roc_auc_score
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator


In [ ]:
DATA_DIR = BASE_PATH / "data" / "GDSC2"
PROCESSED_DIR = BASE_PATH / "data" / "processed"
SPLITS_DIR = BASE_PATH / "data" / "splits"
RESULTS_DIR = BASE_PATH / "results" / "random_forest"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

COL_CELL_LINE = "cell_line_name"
COL_DRUG = "drug_name"
COL_IC50 = "LN_IC50"
COL_CELLOSAURUS = "cellosaurus_id"
COL_TISSUE = "tissue"

SPLIT_TYPES = ["lpo", "lco", "ldo", "lto"]
TOP_K_FEATURES = 1000
RANDOM_STATE = 42

RF_PARAMS = dict(n_estimators=500, max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE)


## Load response pairs + splits (from notebook 04)

In [ ]:
pairs = pd.read_parquet(PROCESSED_DIR / "response_pairs.parquet")

with open(SPLITS_DIR / "splits.json") as f:
    folds = json.load(f)

with open(SPLITS_DIR / "holdout_groups.json") as f:
    holdout_groups = json.load(f)

print(f"{len(pairs)} pairs loaded")
for fd in folds:
    print(f"fold {fd['fold']}: train={len(fd['train']):,} | "
          f"lco_test={len(fd['lco_test']):,} | ldo_test={len(fd['ldo_test']):,} | "
          f"lto_test={len(fd['lto_test']):,} | lpo_test={len(fd['lpo_test']):,}")


## Load omics + drug fingerprints

Dedupe `cellosaurus_id` immediately after load (both RNA and protein have known duplicate rows). Protein gets `fillna(0)` for missing values -- BDRN-validated choice.

In [ ]:
rna = pd.read_csv(DATA_DIR / "gene_expression.csv", index_col=0)
protein = pd.read_csv(DATA_DIR / "proteomics.csv", index_col=0)
drug_smiles = pd.read_csv(DATA_DIR / "drug_smiles.csv")

rna = rna[~rna.index.duplicated(keep="first")].iloc[:, 1:]
protein = protein[~protein.index.duplicated(keep="first")].fillna(0)

OMICS = {"rna": rna, "protein": protein}

print(f"RNA: {rna.shape[1]} genes / {rna.shape[0]} cell lines")
print(f"Protein: {protein.shape[1]} proteins / {protein.shape[0]} cell lines")


In [ ]:
def build_drug_fingerprints(drug_smiles_df: pd.DataFrame, radius: int = 2, n_bits: int = 2048) -> dict:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = {}
    for _, row in drug_smiles_df.iterrows():
        mol = Chem.MolFromSmiles(row["canonical_smiles"])
        if mol is None:
            continue
        fp = generator.GetFingerprint(mol)
        fps[row[COL_DRUG]] = np.array(fp, dtype=np.float32)
    return fps


drug_fp = build_drug_fingerprints(drug_smiles)
print(f"Fingerprints: {len(drug_fp)} / {drug_smiles[COL_DRUG].nunique()} drugs")


## Evaluation metric

Same `evaluateMT` used in notebooks 09 and 10 -- NaN-safe for constant-prediction cases (e.g. a naive fallback, or a degenerate RF leaf).

In [ ]:
def evaluateMT(target, pred, threshold=None):
    """
    threshold: cutoff for binarizing target into sensitive/resistant for ROC-AUC.
    Defaults to median of target if not provided.
    """
    variance_real = np.var(target)
    variance_pred = np.var(pred)
    mses = ((target - pred) ** 2).mean(axis=0)
    rmse = np.sqrt(mses)

    pred_is_constant = np.isclose(variance_pred, 0)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if pred_is_constant:
            correlation, corr_p_value = np.nan, np.nan
            spearman_corr, spearman_p = np.nan, np.nan
            slope, intercept, r_value, lr_p_value, std_err = (np.nan,) * 5
        else:
            correlation, corr_p_value = pearsonr(target, pred)
            spearman_corr, spearman_p = spearmanr(target, pred)
            slope, intercept, r_value, lr_p_value, std_err = linregress(pred, target)

    r2 = r2_score(target, pred)

    if threshold is None:
        threshold = np.median(target)
    target_binary = (target < threshold).astype(int)
    if len(np.unique(target_binary)) == 2 and not pred_is_constant:
        roc_auc = roc_auc_score(target_binary, -pred)
    else:
        roc_auc = np.nan

    results = {
        'Correlation': round(correlation, 2) if not np.isnan(correlation) else correlation,
        'Corr p-value': round(corr_p_value, 4) if not np.isnan(corr_p_value) else corr_p_value,
        'Spearman': round(spearman_corr, 2) if not np.isnan(spearman_corr) else spearman_corr,
        'MSE': round(mses, 2),
        'RMSE': round(rmse, 2),
        'R2': round(r2, 2),
        'ROC-AUC': round(roc_auc, 2) if not np.isnan(roc_auc) else roc_auc,
        'Slope': round(slope, 2) if not np.isnan(slope) else slope,
        'Standard error': round(std_err, 2) if not np.isnan(std_err) else std_err,
        'Variance Real': round(variance_real, 2),
        'Variance Pred': round(variance_pred, 2)
    }
    return pd.DataFrame([results])


## Feature selection + matrix builders

Top-variance feature selection is computed from **train-fold cell lines only** (not the full omics table) to avoid any leakage from val/test cell lines into the selected gene/protein set. Selection happens on the compact per-cell-line table, before expanding to per-pair rows -- expanding first causes the ~11GB memory blowup hit in earlier sessions.

In [ ]:
def select_top_variance_cols(train_idx: np.ndarray, arm: str, k: int = TOP_K_FEATURES) -> pd.Index:
    """Top-k variance columns for an omics arm, computed from train-fold cell lines only."""
    train_cells = pairs.loc[train_idx, COL_CELLOSAURUS].unique()
    compact = OMICS[arm].loc[OMICS[arm].index.intersection(train_cells)]
    return compact.var(axis=0).sort_values(ascending=False).index[:k]


def build_rf_matrix(idx: np.ndarray, components: list[str], top_cols: dict) -> tuple[np.ndarray, np.ndarray]:
    """components: subset of ["rna", "protein", "drug"]."""
    sub = pairs.loc[idx]
    parts = []
    if "rna" in components:
        parts.append(OMICS["rna"][top_cols["rna"]].loc[sub[COL_CELLOSAURUS]].to_numpy())
    if "protein" in components:
        parts.append(OMICS["protein"][top_cols["protein"]].loc[sub[COL_CELLOSAURUS]].to_numpy())
    if "drug" in components:
        parts.append(np.vstack([drug_fp[d] for d in sub[COL_DRUG]]))

    X = np.hstack(parts).astype(np.float32)
    y = sub[COL_IC50].to_numpy().astype(np.float32)

    assert not np.isnan(X).any(), f"NaNs in features for components={components}"
    assert not np.isnan(y).any(), "NaNs in target -- drop these rows upstream, don't impute"

    return X, y


## RandomForest fit + evaluate, per arm

One RandomForest is fit per fold (on the shared train set), then evaluated on all 4 test splits from that same fold -- consistent with the new `splits.json` structure (one train, four tests).

In [ ]:
def run_rf_arm(arm_name: str, components: list[str]) -> list[pd.DataFrame]:
    """Fits one RandomForest per fold for this arm, evaluates on all 4 test splits."""
    results = []
    for fold_i, fold in enumerate(folds):
        start = time.time()
        train_idx = np.array(fold["train"])

        top_cols = {}
        if "rna" in components:
            top_cols["rna"] = select_top_variance_cols(train_idx, "rna")
        if "protein" in components:
            top_cols["protein"] = select_top_variance_cols(train_idx, "protein")

        X_train, y_train = build_rf_matrix(train_idx, components, top_cols)

        model = RandomForestRegressor(**RF_PARAMS)
        model.fit(X_train, y_train)

        for split_type in SPLIT_TYPES:
            test_idx = np.array(fold[f"{split_type}_test"])
            X_test, y_test = build_rf_matrix(test_idx, components, top_cols)
            y_pred = model.predict(X_test)

            df = evaluateMT(y_test, y_pred)
            df.insert(0, "Fold", fold_i)
            df.insert(1, "Split", split_type.upper())
            df.insert(2, "Arm", arm_name)
            results.append(df)

        elapsed = time.time() - start
        print(f"[{arm_name}] fold {fold_i}: trained on {len(train_idx):,} pairs, "
              f"{len(components)} component(s) -- {elapsed:.1f}s")

    return results


### RNA-only

In [ ]:
rna_results = run_rf_arm("RNA", ["rna"])

### Protein-only

In [ ]:
protein_results = run_rf_arm("Protein", ["protein"])

### RNA + drug fingerprint

In [ ]:
rna_drug_results = run_rf_arm("RNA+Drug", ["rna", "drug"])

### Protein + drug fingerprint

In [ ]:
protein_drug_results = run_rf_arm("Protein+Drug", ["protein", "drug"])

## Combine results

In [ ]:
rf_results_df = pd.concat(
    rna_results + protein_results + rna_drug_results + protein_drug_results,
    ignore_index=True
)
rf_results_df


## Summary: mean across folds, per arm x split

In [ ]:
summary_df = (
    rf_results_df
    .groupby(["Arm", "Split"])[["Correlation", "Spearman", "RMSE", "R2", "ROC-AUC"]]
    .mean()
    .round(3)
    .reset_index()
)
summary_df


## Save results

In [ ]:
rf_results_df.to_parquet(RESULTS_DIR / "random_forest_per_fold.parquet")
summary_df.to_csv(RESULTS_DIR / "random_forest_summary.csv", index=False)
print(f"Saved to {RESULTS_DIR}")
